# CPython String Internals, Unicode Architecture, and Interview Guide

## 1. THEORY LAYER
### Origin and Motivation
Python strings represent text sequences. In Python 3, all strings are sequences of Unicode code points (no raw bytes strings exist).

### Memory Model
CPython uses PEP 393 (Flexible String Representation). Instead of allocating 4 bytes per character for all strings, it checks the max code point and allocates:
- Latin-1 (1 byte per char) if max code point <= U+00FF.
- UCS-2 (2 bytes per char) if max code point <= U+FFFF.
- UCS-4 (4 bytes per char) otherwise.

---

## 2. CPYTHON INTERNAL LAYER
### C struct Definition (`unicodeobject.h`)
```c
typedef struct {
    PyObject_HEAD
    Py_ssize_t length;          // Length of string in characters
    Py_hash_t hash;             // Hash value (-1 if not computed)
    struct {
        unsigned int interned:2; // Interned status
        unsigned int kind:3;     // 1=1byte, 2=2bytes, 4=4bytes
        unsigned int compact:1;
    } state;
} PyASCIIObject;
```
- ASCII strings of length 1 are cached permanently.
- Valid variable identifier strings are interned automatically by CPython during bytecode compilation to speed up namespace lookups.

---


In [ ]:
import sys

print("=" * 70)
print("PEP 393 Memory Scaling Demos")
print("=" * 70)

# latin-1
s1 = "abc"
# CJK (BMP)
s2 = "中文"
# Emoji (UCS-4)
s3 = "hi😀"

print(f"Latin-1: '{s1}' | Size: {sys.getsizeof(s1)} bytes")
print(f"UCS-2:   '{s2}' | Size: {sys.getsizeof(s2)} bytes")
print(f"UCS-4:   '{s3}' | Size: {sys.getsizeof(s3)} bytes")



---
## 3. COMPLETE METHOD REFERENCE

### 3.1 `split(sep=None, maxsplit=-1)`
- **Syntax**: `str.split()`
- **Time Complexity**: $O(n)$.


In [ ]:
print("\n=== Method: split ===")
s = "a  b c"
print(f"split() -> {s.split()}")
print(f"split(' ') -> {s.split(' ')} (preserves gaps)")



### 3.2 `join(iterable)`
- **Syntax**: `str.join(iterable)`
- **Time Complexity**: $O(k)$ where $k$ is total length of output.


In [ ]:
print("\n=== Method: join ===")
parts = ["usr", "local", "bin"]
print(f"join -> {'/'.join(parts)}")



### 3.3 `replace(old, new[, count])`
- **Syntax**: `str.replace()`
- **Time Complexity**: $O(n)$.


In [ ]:
print("\n=== Method: replace ===")
s = "banana"
print(f"replace -> {s.replace('a', 'o')}")



### 3.4 `find(sub[, start[, end]])`
- **Syntax**: `str.find(sub)`
- **Behavior**: Returns index of first match, or -1 if not found.


In [ ]:
print("\n=== Method: find ===")
s = "hello"
print(f"find('lo') -> {s.find('lo')}")
print(f"find('z') -> {s.find('z')}")



### 3.5 `startswith(prefix[, start[, end]])`
- **Syntax**: `str.startswith()`
- **Behavior**: Supports string or tuple of options.


In [ ]:
print("\n=== Method: startswith ===")
s = "document.pdf"
print(f"startswith -> {s.startswith(('doc', 'txt'))}")



---
## 4. IMPLEMENTATION LAYER
### Level 1: Pure Python String Reimplementation (Mock ASCII String class)


In [ ]:
class CustomASCIIString:
    """Mock representation of a custom immutable ASCII String in Python."""
    def __init__(self, char_list):
        self._chars = [ord(c) for c in char_list] # Store as ascii codes
        self._hash = None

    def __getitem__(self, index):
        return chr(self._chars[index])

    def __len__(self):
        return len(self._chars)

    def __hash__(self):
        if self._hash is None:
            # Simple FNV-1a like hashing logic
            h = 2166136261
            for code in self._chars:
                h = (h ^ code) * 16777619
                h = h & 0xffffffff
            self._hash = h
        return self._hash

    def __eq__(self, other):
        return isinstance(other, CustomASCIIString) and self._chars == other._chars

    def __repr__(self):
        return "".join(chr(c) for c in self._chars)

print("\n=== Level 1: Custom ASCII String ===")
s = CustomASCIIString("hello")
print(f"String: {s} | Length: {len(s)} | Hash: {hash(s)}")



### Level 2: Python Built-in Comparison
Verify interning behaviors.


In [ ]:
print("\n=== Level 2: String Interning tests ===")
a = "hello_world"
b = "hello_world"
print(f"Implicitly interned literal: {a is b}")

# Dynamic creation bypasses interning
c = "".join(["hello", "_", "world"])
print(f"Dynamically generated same value is interned: {a is c}")

import sys
d = sys.intern(c)
print(f"Forced interning check: {a is d}")



### Level 3: Advanced Usage Systems — Simple String Parser


In [ ]:
class CSVRowParser:
    """Fast CSV row parser splitting entries and trimming quotes."""
    @staticmethod
    def parse_line(line):
        tokens = line.split(",")
        cleaned = [t.strip().strip('"') for t in tokens]
        return cleaned

print("\n=== Level 3: CSV Row Parser ===")
row = ' "Alice", 30 , "Software Engineer" '
print(f"Parsed line: {CSVRowParser.parse_line(row)}")



---
## 5. EXPERIMENTATION LAYER
### Join vs Loop Concat Performance Benchmark


In [ ]:
print("\n=== Section 5: Join vs Loop Concat Benchmark ===")
import time

N = 10000
# Concatenation via loop
t0 = time.perf_counter()
s = ""
for i in range(N):
    s += "a"
t_loop = (time.perf_counter() - t0) * 1000

# Concatenation via join
t0 = time.perf_counter()
parts = ["a"] * N
s = "".join(parts)
t_join = (time.perf_counter() - t0) * 1000

print(f"Loop concat: {t_loop:.2f} ms")
print(f"Join concat: {t_join:.2f} ms")
print(f"Speedup:     {t_loop / (t_join + 1e-10):.1f}x")



---
## 6. VISUALIZATION LAYER
```
PEP 393 Flexible representations:
String Content   ->   State (Kind)   ->   Bytes/Char
"hello"          ->   1 (Latin-1)    ->   1 byte
"你好"           ->   2 (UCS-2)      ->   2 bytes
"hi😀"           ->   4 (UCS-4)      ->   4 bytes
```
---
## 7. INTERVIEW MASTER LAYER

### 50 Scenario-Based Interview Q&As (Summary Selection)

1. **Why are strings immutable?**
   - Immutability allows strings to be hashable (usable as dict keys), allows compiler optimizations (constant folding, interning), and ensures thread safety.
2. **What are the three storage formats of PEP 393?**
   - 1 byte/char (Latin-1), 2 bytes/char (UCS-2), 4 bytes/char (UCS-4).
3. **Explain compile-time string constant folding.**
   - When Python compiles source code to bytecode, expressions like `"a" + "b"` are precalculated to `"ab"` by the peephole optimizer.
